# Create SweCRIS Sweep Awards (multi-funder)

Creates awards for the Swedish funders covered by SweCRIS (Sweden's national research-grants registry, CC0) that are not already first-party ingests in OpenAlex. ~1.2K grants across 5 funders.

**Prerequisites:**
- Run `scripts/local/swecris_to_s3.py` to download and upload the data first.

**Data source:** https://swecris-api.vr.se (SweCRIS API)
**S3 location:** `s3a://openalex-ingest/awards/swecris/swecris_projects.parquet`

**MULTI-FUNDER source (runbook §2.3.2):** each parquet row carries `openalex_funder_id`; the transform joins per row. Funders in this sweep:

| SweCRIS org nr | Funder | OpenAlex |
|---|---|---|
| 202100-2585 | Swedish National Space Agency (Rymdstyrelsen) | F4320321031 |
| 202100-1975 | Naturvårdsverket (Swedish EPA) | F4320322579 |
| 802400-4213 | Stiftelsen för Kunskaps- och Kompetensutveckling (KK-stiftelsen) | F4320321759 |
| 202100-0712 | Statens geotekniska institut (SGI) | F4320316858 |
| 802423-4075 | Familjen Kamprads Stiftelse | F4320325984 |

**Excluded** (already ingested elsewhere): Vetenskapsrådet, Vinnova, Formas, Forte, Riksbankens Jubileumsfond (Complete); Energimyndigheten (priority 435, provenance `energimyndigheten`); Östersjöstiftelsen + IFAU (first-party at priorities 327/338).

**Mapping notes:**
- `funder_award_id` = SweCRIS projectId with the per-funder suffix stripped (e.g. `1.1-1708-0499_SGI` → `1.1-1708-0499`).
- `amount` = `fundingsSek`, **SEK** (all five are Swedish funders), zeros treated as not-published.
- PI from SweCRIS peopleList (given/family split with the canonical split_name helper in the script).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.swecris_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/swecris/swecris_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.swecris_raw;

In [ ]:
%sql
-- Step 1.5: inspect raw data before transforming
DESCRIBE openalex.awards.swecris_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.swecris_raw LIMIT 5;

In [ ]:
%sql
-- Step 1.6 funder existence check (Path A: expect exactly 5 rows)
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id IN (4320321031, 4320322579, 4320321759, 4320316858, 4320325984);

## Step 2: Create SweCRIS Sweep Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.swecris_awards
USING delta
AS
WITH
swecris_funders AS (
    -- §2.3.2 multi-funder: joined per row on the parquet's openalex_funder_id
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id IN (4320321031, 4320322579, 4320321759, 4320316858, 4320325984)
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(REGEXP_REPLACE(TRIM(g.project_id), '_[A-Za-z]+$', ''))))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title_english), ''), NULLIF(TRIM(g.title), '')) as display_name,
        COALESCE(NULLIF(TRIM(g.abstract_english), ''), NULLIF(TRIM(g.abstract), '')) as description,
        f.funder_id,
        REGEXP_REPLACE(TRIM(g.project_id), '_[A-Za-z]+$', '') as funder_award_id,
        NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) as amount,
        'SEK' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        CASE
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%fellow%' THEN 'fellowship'
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%stipend%' THEN 'fellowship'
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%position%' THEN 'fellowship'
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%infrastructure%' THEN 'infrastructure'
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%project%' THEN 'research'
            ELSE 'grant'
        END as funding_type,
        NULLIF(TRIM(g.type_of_award), '') as funder_scheme,
        'swecris' as provenance,
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
        CASE
            WHEN g.pi_family_name IS NOT NULL AND TRIM(g.pi_family_name) != '' THEN
                struct(
                    NULLIF(TRIM(g.pi_given_name), '') as given_name,
                    TRIM(g.pi_family_name) as family_name,
                    NULLIF(TRIM(g.pi_orcid), '') as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        NULLIF(TRIM(g.coordinating_organisation), '') as name,
                        'Sweden' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
        CONCAT('https://www.vr.se/swecris#/project/', TRIM(g.project_id)) as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(REGEXP_REPLACE(TRIM(g.project_id), '_[A-Za-z]+$', ''))))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.swecris_raw g
    JOIN swecris_funders f
      ON f.funder_id = TRY_CAST(g.openalex_funder_id AS BIGINT)
    WHERE g.project_id IS NOT NULL AND TRIM(g.project_id) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'swecris' AND priority = 437;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    437 as priority  -- SweCRIS sweep priority
FROM openalex.awards.swecris_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_swecris_awards FROM openalex.awards.swecris_awards;

In [ ]:
%sql
-- 6.5 funder split (multi-funder source: every funder must have a plausible count)
SELECT funder.display_name, funder_id, COUNT(*) as n,
       COUNT(amount) as has_amount, COUNT(lead_investigator) as has_pi,
       ROUND(SUM(amount)/1e6, 1) as total_msek
FROM openalex.awards.swecris_awards
GROUP BY funder.display_name, funder_id ORDER BY n DESC;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_abstract,
    COUNT(amount) as has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    COUNT(lead_investigator) as has_pi,
    MIN(amount) as min_amount,
    ROUND(AVG(amount), 0) as avg_amount,
    MAX(amount) as max_amount
FROM openalex.awards.swecris_awards;

In [ ]:
%sql
-- 6.4a PI frequency check
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.swecris_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;